In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))


# ── shared/mlfp05/ex_2.py ──
"""
Shared utilities for MLFP05 Exercise 2 — CNNs on CIFAR-10.

Common infrastructure used by all technique files:
  - CIFAR-10 data loading and normalisation
  - Device detection and precision settings
  - ExperimentTracker / ModelRegistry setup
  - LightningModule wrapper for training
  - Visualisation helpers
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pytorch_lightning as pl
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.engines.model_registry import LocalFileArtifactStore
from kailash_ml.types import MetricSpec

# ── Environment & Reproducibility ────────────────────────────────────
setup_environment()
torch.manual_seed(42)
np.random.seed(42)
pl.seed_everything(42, workers=True)

DEVICE = get_device()
# Notebook-safe path resolution: __file__ is undefined when this module is
# inlined into a Colab notebook by scripts/generate_selfcontained_notebook.py.
try:
    _HERE = Path(__file__).resolve()
    REPO_ROOT = _HERE.parents[2]
    ARTIFACT_DIR = _HERE.parent
except NameError:
    REPO_ROOT = Path.cwd()
    ARTIFACT_DIR = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"

# The ModelRegistry stores artifact files (model.pkl, model.onnx) in an
# ArtifactStore. We construct the store explicitly so exercises can attach
# an ONNX serving artifact to a registered version (see attach_onnx_artifact).
ARTIFACT_STORE = LocalFileArtifactStore(".kailash_ml/artifacts")

N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

# ── CIFAR-10 class labels (used for visualisation and apply sections) ─
CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Per-channel normalisation statistics (CIFAR-10 population)
CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)


# ═══════════════════════════════════════════════════════════════════════
# Data Loading
# ═══════════════════════════════════════════════════════════════════════
def load_cifar10() -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    DataLoader,
    DataLoader,
]:
    """Load and normalise the full CIFAR-10 dataset.

    Returns:
        X_train, y_train, X_val, y_val, train_loader, val_loader
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_train = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_train = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))],
        dtype=torch.long,
    )
    X_val = torch.stack([val_set[i][0] for i in range(len(val_set))])
    y_val = torch.tensor(
        [val_set[i][1] for i in range(len(val_set))],
        dtype=torch.long,
    )

    # Per-channel normalisation
    X_train = (X_train - CIFAR_MEAN) / CIFAR_STD
    X_val = (X_val - CIFAR_MEAN) / CIFAR_STD

    train_ds = TensorDataset(X_train, y_train)
    val_ds = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    print(
        f"CIFAR-10: train {tuple(X_train.shape)}, val {tuple(X_val.shape)}, "
        f"classes={N_CLASSES}: {CLASS_NAMES}"
    )
    return X_train, y_train, X_val, y_val, train_loader, val_loader


# ═══════════════════════════════════════════════════════════════════════
# Kailash Engine Setup
# ═══════════════════════════════════════════════════════════════════════
async def setup_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Initialise ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry.

    Returns:
        conn, tracker, experiment_name, registry, has_registry
    """
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker and
    # ModelRegistry use incompatible _kml_model_versions schemas. Route them
    # to separate sqlite files until upstream fixes the conflict.
    db = f"sqlite:///{db_name}"
    registry_db_name = (
        db_name.replace(".db", "_registry.db")
        if db_name.endswith(".db")
        else db_name + "_registry.db"
    )
    registry_db = f"sqlite:///{registry_db_name}"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn, artifact_store=ARTIFACT_STORE)
    return conn, tracker, "m5_cnns", registry, True


def init_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Sync wrapper for setup_engines."""
    return asyncio.run(setup_engines(db_name))


# ═══════════════════════════════════════════════════════════════════════
# Lightning Training Infrastructure
# ═══════════════════════════════════════════════════════════════════════
class LitCNN(pl.LightningModule):
    """Lightning wrapper for any nn.Module classifier.

    Tracks per-epoch training loss and validation accuracy for later
    logging to ExperimentTracker and visualisation with ModelVisualizer.
    """

    def __init__(self, model: nn.Module, lr: float = 1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        self.train_losses: list[float] = []
        self.val_accs: list[float] = []
        self._batch_losses: list[float] = []
        self._val_correct = 0
        self._val_total = 0

    def training_step(self, batch, batch_idx):
        del batch_idx  # Lightning protocol arg; not used here
        x, y = batch
        logits = self.model(x)
        loss = F.cross_entropy(logits, y)
        self._batch_losses.append(loss.item())
        return loss

    def on_train_epoch_end(self):
        if self._batch_losses:
            self.train_losses.append(float(np.mean(self._batch_losses)))
            self._batch_losses = []

    def validation_step(self, batch, batch_idx):
        del batch_idx  # Lightning protocol arg; not used here
        x, y = batch
        logits = self.model(x)
        self._val_correct += int((logits.argmax(dim=-1) == y).sum().item())
        self._val_total += int(y.size(0))

    def on_validation_epoch_end(self):
        if self._val_total > 0:
            self.val_accs.append(self._val_correct / self._val_total)
            self._val_correct = 0
            self._val_total = 0

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=self.lr)


def get_precision_setting() -> str:
    """Return the optimal Lightning precision string for the current backend.

    Routes through ``kailash_ml.device()`` so MPS / CUDA / ROCm / Intel XPU /
    CPU are all selected by the same policy the rest of the platform uses.
    BackendInfo.precision is the canonical answer ("16-mixed" on Apple MPS
    + Tensor-Core GPUs, "32" on older GPUs and CPU).
    """
    import kailash_ml as km

    backend = km.device()
    print(f"  {backend.backend} detected -- using {backend.precision} precision")
    return backend.precision


def get_accelerator() -> str:
    """Return the Lightning accelerator string for the current backend."""
    import kailash_ml as km

    return km.device().accelerator


PRECISION = get_precision_setting()
ACCELERATOR = get_accelerator()


async def train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Train a CNN with Lightning and log metrics to ExperimentTracker.

    Uses the kailash-ml 1.1.1 ``tracker.track(...)`` async context manager.
    On normal exit the run is marked FINISHED; on exception it is marked FAILED.
    """
    lit = LitCNN(model, lr=lr)
    trainer = pl.Trainer(
        max_epochs=epochs,
        accelerator=ACCELERATOR,
        precision=PRECISION,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        enable_checkpointing=False,
    )

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "architecture": name,
                "lr": str(lr),
                "epochs": str(epochs),
                "batch_size": str(BATCH_SIZE),
                "dataset_size": str(len(train_loader.dataset)),
                "precision": PRECISION,
                "accelerator": ACCELERATOR,
            }
        )

        trainer.fit(lit, train_loader, val_loader)

        for epoch_idx, loss in enumerate(lit.train_losses):
            await run.log_metric("train_loss", loss, step=epoch_idx + 1)
        for epoch_idx, acc in enumerate(lit.val_accs):
            await run.log_metric("val_accuracy", acc, step=epoch_idx + 1)

        await run.log_metrics(
            {
                "final_train_loss": lit.train_losses[-1],
                "final_val_accuracy": lit.val_accs[-1],
            }
        )

    print(
        f"  [{name}] final train loss={lit.train_losses[-1]:.4f}  "
        f"val acc={lit.val_accs[-1]:.3f}"
    )
    return lit.train_losses, lit.val_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        train_model_async(
            model,
            name,
            tracker,
            exp_name,
            train_loader,
            val_loader,
            lr,
            epochs,
        )
    )


# ═══════════════════════════════════════════════════════════════════════
# Model Registration
# ═══════════════════════════════════════════════════════════════════════
async def register_model_async(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Register a trained model with metrics in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="val_accuracy", value=final_acc),
            MetricSpec(name="epochs", value=float(epochs)),
            MetricSpec(name="batch_size", value=float(BATCH_SIZE)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={final_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Sync wrapper for register_model_async."""
    return asyncio.run(
        register_model_async(registry, name, model, final_loss, final_acc, epochs)
    )


# ═══════════════════════════════════════════════════════════════════════
# Visualisation Helpers
# ═══════════════════════════════════════════════════════════════════════
def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance.

    Delegates to the canonical factory in shared.mlfp05 — the P2
    experimental notice is acknowledged there.
    """

    return _canonical()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
    x_label: str = "Epoch",
    y_label: str = "Value",
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(
        metrics=metrics,
        x_label=x_label,
        y_label=y_label,
    )
    fig.write_html(str(output_path))
    print(f"  Saved plot: {output_path}")


def count_parameters(model: nn.Module) -> int:
    """Count total trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters())


def denormalise_cifar(img_tensor: torch.Tensor) -> torch.Tensor:
    """Reverse CIFAR-10 normalisation for display.

    Args:
        img_tensor: shape (C, H, W) or (B, C, H, W), normalised

    Returns:
        Tensor with pixel values clipped to [0, 1]
    """
    if img_tensor.dim() == 3:
        mean = CIFAR_MEAN.squeeze(0)
        std = CIFAR_STD.squeeze(0)
    else:
        mean = CIFAR_MEAN
        std = CIFAR_STD
    return (img_tensor * std + mean).clamp(0, 1)


# ═══════════════════════════════════════════════════════════════════════
# Serving Helpers (ONNX export + InferenceServer)
# ═══════════════════════════════════════════════════════════════════════
IMAGE_SHAPE = (3, 32, 32)
N_PIXELS = 3 * 32 * 32


class FlatImageAdapter(nn.Module):
    """Wrap an image classifier so it accepts flat pixel rows.

    InferenceServer's ONNX runtime turns each request record into ONE row of
    a 2-D float array, so the exported graph must take ``(batch, 3072)``.
    This adapter reshapes rows back to ``(batch, 3, 32, 32)`` before calling
    the wrapped CNN. ``predict(X)`` is the method ``OnnxBridge.validate``
    calls to get the native (PyTorch) outputs for comparison.
    """

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x.reshape(-1, *IMAGE_SHAPE))

    def predict(self, X) -> np.ndarray:
        device = next(self.model.parameters()).device
        with torch.no_grad():
            x = torch.as_tensor(np.asarray(X), dtype=torch.float32, device=device)
            return self.forward(x).cpu().numpy()


def images_to_records(images: torch.Tensor) -> list[dict[str, float]]:
    """Turn a batch of images into InferenceServer request records.

    One record per image, one key per pixel. Keys are zero-padded so the
    record order is the pixel order the ONNX graph expects.
    """
    rows = images.reshape(len(images), -1).cpu().tolist()
    return [{f"px{j:04d}": float(v) for j, v in enumerate(row)} for row in rows]


def attach_onnx_artifact(name: str, version: int, onnx_path: Path) -> None:
    """Store an exported ONNX file as ``model.onnx`` for a registered version.

    ``ModelRegistry.register_model`` saves the training artifact as
    ``model.pkl``; ``InferenceServer.from_registry(..., runtime="onnx")``
    loads ``model.onnx`` from the same ArtifactStore.

    OnnxBridge writes large weights to a sibling ``<file>.onnx.data`` file
    (ONNX external data); storing only the ``.onnx`` bytes would serve a model
    without its weights, so it is stored as ONE self-contained protobuf.
    """
    import onnx

    model_proto = onnx.load(str(onnx_path))  # pulls in any .onnx.data weights
    asyncio.run(
        ARTIFACT_STORE.save(name, version, model_proto.SerializeToString(), "model.onnx")
    )


# ── shared/mlfp05/ex_8.py ──
"""
Shared infrastructure for Exercise 8 — Reinforcement Learning.

Contains: CartPole setup, reward plotting helpers, ExperimentTracker/ModelRegistry
setup, replay buffer + DQN network, evaluation utilities, and the RL
diagnostic checkpoint (kailash_ml RLDiagnostics).
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
import random
from collections import deque
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

import gymnasium as gym
from gymnasium import spaces

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex8_reinforcement_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# CARTPOLE ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════


def make_cartpole() -> tuple[gym.Env, int, int]:
    """Create CartPole-v1 and return (env, obs_dim, n_actions)."""
    env = gym.make("CartPole-v1")
    obs_space = env.observation_space
    act_space = env.action_space
    assert (
        isinstance(obs_space, spaces.Box) and obs_space.shape is not None
    ), f"CartPole obs space expected gymnasium Box, got {type(obs_space).__name__}"
    assert isinstance(
        act_space, spaces.Discrete
    ), f"CartPole action space expected Discrete, got {type(act_space).__name__}"
    obs_dim = obs_space.shape[0]
    n_actions = int(act_space.n)
    print(f"CartPole-v1  obs_dim={obs_dim}  n_actions={n_actions}")
    return env, obs_dim, n_actions


def make_pendulum() -> tuple[gym.Env, int, int, float]:
    """Create Pendulum-v1 and return (env, obs_dim, act_dim, act_limit).

    Pendulum is the canonical CONTINUOUS control benchmark: 3-D state
    (cos/sin angle + angular velocity), one continuous torque in
    [-2, 2]. Rewards are negative (best ~= 0 when upright and still);
    a random policy averages about -1200 per episode.
    """
    env = gym.make("Pendulum-v1")
    obs_space = env.observation_space
    act_space = env.action_space
    assert (
        isinstance(obs_space, spaces.Box) and obs_space.shape is not None
    ), f"Pendulum obs space expected Box, got {type(obs_space).__name__}"
    assert (
        isinstance(act_space, spaces.Box) and act_space.shape is not None
    ), f"Pendulum action space expected Box, got {type(act_space).__name__}"
    obs_dim = obs_space.shape[0]
    act_dim = act_space.shape[0]
    act_limit = float(act_space.high[0])
    print(
        f"Pendulum-v1  obs_dim={obs_dim}  act_dim={act_dim}  "
        f"act_limit={act_limit}"
    )
    return env, obs_dim, act_dim, act_limit


# ════════════════════════════════════════════════════════════════════════
# CONTINUOUS REPLAY BUFFER — shared by DDPG and SAC (float actions)
# ════════════════════════════════════════════════════════════════════════


class ContinuousReplayBuffer:
    """Fixed-size buffer for continuous actions (float tensors, not long)."""

    def __init__(self, capacity: int = 50_000):
        self.buffer: deque = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(list(self.buffer), batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32, device=device),
            torch.tensor(np.array(actions), dtype=torch.float32, device=device),
            torch.tensor(rewards, dtype=torch.float32, device=device),
            torch.tensor(np.array(next_states), dtype=torch.float32, device=device),
            torch.tensor(dones, dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open the kailash-ml tracker + registry. Returns a 5-tuple."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_rl.db"
    registry_db = "sqlite:///mlfp05_rl_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_reinforcement_learning", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# REPLAY BUFFER — shared by DQN and custom env training
# ════════════════════════════════════════════════════════════════════════


class ReplayBuffer:
    """Fixed-size buffer storing (state, action, reward, next_state, done)."""

    def __init__(self, capacity: int = 10_000):
        self.buffer: deque = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(list(self.buffer), batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32, device=device),
            torch.tensor(actions, dtype=torch.long, device=device),
            torch.tensor(rewards, dtype=torch.float32, device=device),
            torch.tensor(np.array(next_states), dtype=torch.float32, device=device),
            torch.tensor(dones, dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)


# ════════════════════════════════════════════════════════════════════════
# DQN NETWORK — shared by DQN training and custom env training
# ════════════════════════════════════════════════════════════════════════


class DQN(nn.Module):
    """Deep Q-Network: maps state -> Q-value for each action."""

    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ════════════════════════════════════════════════════════════════════════
# EVALUATION UTILITY
# ════════════════════════════════════════════════════════════════════════


def evaluate_policy(env: gym.Env, policy_fn, n_episodes: int = 30) -> list[float]:
    """Evaluate a policy function over n_episodes. Returns list of total rewards."""
    eval_returns: list[float] = []
    for i in range(n_episodes):
        state, _ = env.reset(seed=1000 + i)
        total = 0.0
        done = False
        while not done:
            action = policy_fn(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total += float(reward)
            done = terminated or truncated
        eval_returns.append(total)
    return eval_returns


# ════════════════════════════════════════════════════════════════════════
# REWARD PLOTTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def moving_average(xs: list[float], window: int = 10) -> list[float]:
    """Smooth a time series with a rolling mean."""
    if len(xs) < window:
        return xs
    arr = np.asarray(xs, dtype=np.float32)
    kernel = np.ones(window, dtype=np.float32) / window
    return list(np.convolve(arr, kernel, mode="valid"))


def plot_reward_curve(
    viz: ModelVisualizer,
    rewards: list[float],
    title: str,
    filename: str,
    window: int = 20,
    x_label: str = "Episode",
    y_label: str = "Reward",
) -> None:
    """Plot a reward curve with moving average and save to HTML."""
    metrics = {
        f"{title} reward": rewards,
        f"{title} moving avg ({window})": moving_average(rewards, window),
    }
    fig = viz.training_history(metrics=metrics, x_label=x_label, y_label=y_label)
    out_path = OUTPUT_DIR / filename
    fig.write_html(str(out_path))
    print(f"  Saved: {out_path}")


# ════════════════════════════════════════════════════════════════════════
# RL DIAGNOSTIC CHECKPOINT — kailash_ml.diagnostics.RLDiagnostics
# ════════════════════════════════════════════════════════════════════════
# The DL Prescription Pad (gradient flow / dead neurons / loss trend) is
# built for supervised batches. RL has its own instrument in kailash-ml:
# RLDiagnostics (also what `km.diagnose(algo, kind="rl")` returns). It
# installs no hooks — you feed it the training history you recorded and
# `report()` summarises it. Its automated finding is REWARD COLLAPSE: a
# CRIT alert when the latest reward falls to <10% of the peak after a
# >=50% drop over the rolling window.


def rl_diagnostic_checkpoint(
    title: str,
    algo: str,
    rewards: list[float],
    *,
    lengths: list[int] | None = None,
    q_losses: list[float] | None = None,
    policy_losses: list[float] | None = None,
    value_losses: list[float] | None = None,
    entropies: list[float] | None = None,
    window: int = 20,
) -> dict:
    """Feed a recorded RL training history to RLDiagnostics and print it.

    ``rewards`` holds one entry per episode (DQN) or per PPO iteration
    (the iteration's mean episode return). ``policy_losses`` /
    ``value_losses`` / ``entropies`` are per PPO iteration; ``q_losses``
    per DQN episode (entries of 0.0 = no gradient step yet are skipped).
    Returns the ``report()`` dict.
    """
    from kailash_ml.diagnostics import RLDiagnostics

    diag = RLDiagnostics(algo=algo, window=window)
    for i, reward in enumerate(rewards):
        length = int(lengths[i]) if lengths is not None else 0
        diag.record_episode(reward=float(reward), length=length)
    if policy_losses is not None:
        for i, loss in enumerate(policy_losses):
            entropy = float(entropies[i]) if entropies is not None else None
            diag.record_policy_update(float(loss), entropy=entropy)
            if value_losses is not None:
                diag.record_value_update(float(value_losses[i]))
    n_q = 0
    for loss in q_losses or []:
        if loss > 0.0:
            diag.record_q_update(float(loss))
            n_q += 1
    report = diag.report()

    metrics = report["metrics"]
    unit = "episodes" if algo == "dqn" else "iterations"
    print("=" * 66)
    print(f"  RL Diagnostics — {title}")
    print("=" * 66)
    print(f"  {unit} recorded:            {metrics['episode_count']}")
    print(
        f"  mean reward (last {min(window, len(rewards))} {unit}): "
        f"{metrics['episode_reward_mean']:.2f}"
    )
    print(f"  peak reward:                {metrics['episode_reward_peak']:.2f}")
    if policy_losses is not None:
        print(f"  policy updates recorded:    {metrics['update_count']}")
    if q_losses is not None:
        print(f"  Q-loss entries recorded:    {n_q}")
    if report["findings"]:
        for finding in report["findings"]:
            print(f"  [{finding['severity']}] {finding['category']}: {finding['message']}")
            print(f"        suggestion: {finding['suggestion']}")
    else:
        print("  findings: none — no reward collapse over the rolling window")
    print("=" * 66)
    return report


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION HELPER
# ════════════════════════════════════════════════════════════════════════


async def _register_rl_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics_dict: dict[str, float],
):
    """Register a single RL policy network in ModelRegistry."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    metric_specs = [MetricSpec(name=k, value=v) for k, v in metrics_dict.items()]
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=metric_specs,
    )
    print(f"  Registered {name}: version={version.version}")
    return version


def register_rl_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics_dict: dict[str, float],
):
    """Sync wrapper for RL model registration."""
    return asyncio.run(_register_rl_model(registry, name, model, metrics_dict))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 Exercise 8.5 — PPO for CONTINUOUS Actions: the Gaussian Policy
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this file, you will be able to:
#   - Explain why a Categorical head cannot express "apply 1.37 Nm of
#     torque" — and what a Gaussian policy head outputs instead
#   - Implement a Gaussian actor: network emits mu(s); a learned log_std
#     carries the exploration noise; log_prob comes from Normal
#   - Reuse the ENTIRE PPO stack (GAE, clipped surrogate, entropy bonus)
#     unchanged — only the distribution changes
#   - Train PPO on Pendulum-v1 (a Box action space) and verify the run
#     beats a measured random-policy baseline
#   - Visualise the exploration noise collapsing as the policy gains
#     confidence (log_std schedule, measured not assumed)
#   - Apply continuous control to HVAC damper control at a data centre
#
# PREREQUISITES: M5/ex_8/02_ppo.py (GAE, clipped surrogate, PPO loop on
#   discrete CartPole). This file changes ONE thing: the action
#   distribution.
# ESTIMATED TIME: ~35 min
#
# ENVIRONMENT: Pendulum-v1 (Gymnasium classic control; 3-D Box state,
#   1-D Box action in [-2, 2]). Rewards are negative; random ~= -1200,
#   a trained policy approaches -150. CPU-budgeted: 25 iterations of
#   1024 steps.
#
# PHASES:
#   1. THEORY  — continuous action spaces and the Gaussian policy
#   2. BUILD   — GaussianActorCritic
#   3. TRAIN   — PPO on Pendulum, tracked with ExperimentTracker
#   4. VISUALISE — reward curve, exploration-noise collapse
#   5. APPLY   — HVAC damper control at a data centre
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from kailash_ml import ModelVisualizer

# Deterministic seeds — RL outcomes must reproduce run to run, or the
# checkpoint below is unverifiable and students chase noise.
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)



## PHASE 1 — THEORY: When the Action Is a Dial, Not a Button


Discrete action (CartPole):  softmax over 2 buttons     (Categorical)
  Continuous action (Pendulum): mean + learned spread on a dial (Normal)

  ONLY THE DISTRIBUTION CHANGES:
    a ~ Normal(mu_theta(s), exp(log_std));  ratio = exp(new_logprob - old)
    GAE and clipping: identical to 02_ppo.py. Two honest adjustments for
    the dial: entropy bonus OFF (the spread IS the entropy) and rewards
    scaled 0.1 for the critic (a units change; reported returns are raw).

  EXPLORATION = the spread. log_std shrinking over training IS the
  policy gaining confidence — we measure it, not assume it.


In [ ]:
# CartPole's action is a BUTTON: push left or push right — a Categorical
# distribution over 2 choices. Pendulum's action is a DIAL: a torque in
# [-2, 2]. A softmax over infinitely many settings is impossible.
#
# THE GAUSSIAN POLICY (the standard answer):
#   The actor outputs the MEAN of a Normal distribution; a separate
#   learned parameter (log_std) controls the spread. Acting is sampling:
#       a ~ Normal(mu_theta(s), std)
#   The log-probability of the sampled action (needed for the PPO ratio)
#   comes from the Normal's log_prob — the same role Categorical played.
#
# WHAT CHANGES FROM DISCRETE PPO: essentially only the distribution object.
#   GAE: unchanged.  Clipped surrogate: unchanged.  Entropy bonus: dropped
#   (coef 0.0) — on a continuous dial the spread IS the entropy term, and
#   paying a bonus for it pins the dial in the noisy regime. We also scale
#   rewards by 0.1 for the critic (a units change, not a reward hack):
#   Pendulum's -1000-scale returns would otherwise dominate the value loss.
#
# EXPLORATION, MEASURED: log_std starts near 0 (std ~ 1 — wild swings of
# the dial) and should SHRINK as the policy converges on a confident mean
# torque. We plot the measured log_std schedule — exploration collapse
# you can see, not a claim in a comment.

print("=" * 70)
print("  PHASE 1 — THEORY: the Gaussian policy for continuous actions")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: GaussianActorCritic


Actor: MLP -> mean torque, squashed by tanh into [-ACT_LIMIT, LIMIT].
    log_std is a FREE parameter (state-independent spread) — the simplest
    honest exploration schedule, and the one whose collapse we plot.
    Critic: separate MLP -> V(s), kept independent for the same reason as
    in 02_ppo.py (value-regression gradients dominate a shared trunk).


Sample an action. Returns (action_array, log_prob, value).


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: Gaussian actor + value critic")
print("=" * 70)

env, obs_dim, act_dim, ACT_LIMIT = make_pendulum()
conn, tracker, exp_name, registry, has_registry = setup_engines()


class GaussianActorCritic(nn.Module):

    def __init__(self, obs_dim: int, act_dim: int, hidden: int = 64):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, act_dim),
        )
        # log_std starts at -0.5 (std ~ 0.6): wide enough to explore the
        # dial, not so wide that early updates are pure noise.
        self.log_std = nn.Parameter(torch.full((act_dim,), -0.5))
        self.critic = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )
        # Small orthogonal policy-head init (gain 0.01): initial mean torque
        # ~ 0, so the critic maps the value landscape before the actor
        # commits to a strategy. The standard PPO initialisation for
        # continuous control (and the difference between converging and
        # plateauing on Pendulum at CPU budgets).
        for m in self.actor:
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        for m in self.critic:
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor[-1].weight, gain=0.01)
        nn.init.zeros_(self.actor[-1].bias)
        nn.init.orthogonal_(self.critic[-1].weight, gain=1.0)
        nn.init.zeros_(self.critic[-1].bias)

    def forward(self, x: torch.Tensor) -> tuple[Normal, torch.Tensor]:
        mu = torch.tanh(self.actor(x)) * ACT_LIMIT
        dist = Normal(mu, self.log_std.exp().expand_as(mu))
        return dist, self.critic(x).squeeze(-1)

    def act(self, state: np.ndarray) -> tuple[np.ndarray, torch.Tensor, torch.Tensor]:
        s = torch.from_numpy(state.astype(np.float32)).to(device)
        dist, value = self.forward(s)
        a = dist.sample()
        return (
            a.cpu().numpy().astype(np.float32),
            dist.log_prob(a).sum(-1).detach(),
            value.detach(),
        )



### Checkpoint 1: shapes, action bounds, log-prob sanity


In [ ]:
_probe = GaussianActorCritic(obs_dim, act_dim).to(device)
_s, _ = env.reset(seed=0)
_a, _lp, _v = _probe.act(_s)
assert _a.shape == (act_dim,), f"action shape {_a.shape} != ({act_dim},)"
assert np.all(np.abs(_a) <= ACT_LIMIT + 1e-6), (
    f"tanh-squashed action {_a} exceeds +/-{ACT_LIMIT}"
)
assert np.isfinite(float(_lp)), "log_prob must be finite"
with torch.no_grad():
    dist_probe, _ = _probe(torch.from_numpy(_s.astype(np.float32)).to(device))
assert abs(float(dist_probe.stddev.mean()) - float(np.exp(-0.5))) < 1e-4, (
    "log_std starts at -0.5 -> initial std must be exp(-0.5) ~ 0.607"
)
_n_params = sum(p.numel() for p in _probe.parameters())
print(f"\nGaussianActorCritic built: {_n_params:,} parameters")
print(f"  action shape ({act_dim},), bounded in [-{ACT_LIMIT}, {ACT_LIMIT}]")
print(f"  initial exploration std = {np.exp(-0.5):.3f} (log_std = -0.5, free parameter)")
print(f"  policy head init: orthogonal gain 0.01 (initial mean torque ~ 0)")
print("\n--- Checkpoint 1 passed --- Gaussian policy verified\n")
del _probe, _s, _a, _lp, _v, dist_probe



## PHASE 3 — TRAIN: PPO on Pendulum (GAE + clip, unchanged)


On-policy rollout for continuous actions (float arrays, not ints).


GAE — identical to 02_ppo.py (the distribution changed, GAE didn't).


PPO loop with the Gaussian head; returns history for plotting.


In [ ]:
print("=" * 70)
print("  PHASE 3 — TRAIN: continuous PPO on Pendulum-v1")
print("=" * 70)

# Measured random baseline FIRST — every improvement claim needs it.
random_returns = evaluate_policy(env, lambda s: env.action_space.sample(),
                                 n_episodes=10)
random_mean = float(np.mean(random_returns))
print(f"  Random policy baseline: {random_mean:.1f} mean return (10 episodes)")

N_ITERS = 240
STEPS_PER_ITER = 1024
GAMMA, LAM, CLIP_EPS, LR = 0.99, 0.95, 0.2, 1e-3
REWARD_SCALE = 0.1  # critic/GAE see scaled rewards; REPORTED returns are raw
# Pendulum's rewards are large and negative (a swing-up costs ~-1000).
# Scaling by 0.1 keeps the critic's MSE in a sane numeric range without
# changing the ordering of policies — a units change, not a reward hack.
# No entropy bonus here (coef 0.0): for a continuous dial the exploration
# spread IS the entropy term, and paying a bonus for it pins the dial in
# the noisy regime — exactly the failure the std schedule is here to catch.


def collect_continuous_trajectory(env, model, max_steps):
    states, actions, log_probs, values, rewards, dones = [], [], [], [], [], []
    state, _ = env.reset(seed=int(np.random.randint(0, 100_000)))
    for _ in range(max_steps):
        action, log_prob, value = model.act(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        states.append(state.astype(np.float32))
        actions.append(action)
        log_probs.append(log_prob)
        values.append(value)
        rewards.append(float(reward))
        done = terminated or truncated
        dones.append(done)
        state = next_state
        if done:  # Pendulum truncates at 200 steps; reset and continue
            state, _ = env.reset(seed=int(np.random.randint(0, 100_000)))
    return states, actions, log_probs, values, rewards, dones


def compute_gae(rewards, values, dones, gamma=GAMMA, lam=LAM):
    advantages = [0.0] * len(rewards)
    gae, next_value = 0.0, 0.0
    for t in reversed(range(len(rewards))):
        nonterminal = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * next_value * nonterminal - float(values[t])
        gae = delta + gamma * lam * nonterminal * gae
        advantages[t] = gae
        next_value = float(values[t])
    returns = [a + float(v) for a, v in zip(advantages, values)]
    return advantages, returns


async def train_continuous_ppo():
    model = GaussianActorCritic(obs_dim, act_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    iter_returns, entropies, actor_losses, critic_losses, std_hist = (
        [], [], [], [], [],
    )

    async with tracker.track(experiment=exp_name, run_name="ppo_pendulum") as run:
        await run.log_params(
            {
                "algorithm": "PPO-continuous",
                "env": "Pendulum-v1",
                "gamma": str(GAMMA),
                "lambda": str(LAM),
                "clip_eps": str(CLIP_EPS),
                "lr": str(LR),
                "steps_per_iter": str(STEPS_PER_ITER),
                "random_baseline_return": f"{random_mean:.1f}",
            }
        )
        for it in range(N_ITERS):
            states, actions, old_lps, values, rewards, dones = (
                collect_continuous_trajectory(env, model, STEPS_PER_ITER)
            )
            scaled_rewards = [r * REWARD_SCALE for r in rewards]
            advantages, returns = compute_gae(scaled_rewards, values, dones)

            s_t = torch.tensor(np.stack(states), dtype=torch.float32, device=device)
            a_t = torch.tensor(np.stack(actions), dtype=torch.float32, device=device)
            old_lp_t = torch.stack(old_lps).to(device)
            adv_t = torch.tensor(advantages, dtype=torch.float32, device=device)
            ret_t = torch.tensor(returns, dtype=torch.float32, device=device)
            adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)

            n = s_t.size(0)
            idxs = np.arange(n)
            it_actor = it_critic = it_entropy = 0.0
            updates = 0
            for _ in range(4):  # PPO epochs over the same rollout
                np.random.shuffle(idxs)
                for start in range(0, n, 256):
                    mb = idxs[start : start + 256]
                    dist, vpred = model(s_t[mb])
                    new_lp = dist.log_prob(a_t[mb]).sum(-1)
                    ratio = torch.exp(new_lp - old_lp_t[mb])
                    surr1 = ratio * adv_t[mb]
                    surr2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv_t[mb]
                    policy_loss = -torch.min(surr1, surr2).mean()
                    value_loss = F.mse_loss(vpred, ret_t[mb])
                    entropy = dist.entropy().sum(-1).mean()
                    # entropy coef 0.0 — see REWARD_SCALE comment above
                    loss = policy_loss + 0.5 * value_loss
                    opt.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                    opt.step()
                    it_actor += policy_loss.item()
                    it_critic += value_loss.item()
                    it_entropy += entropy.item()
                    updates += 1

            # Mean episode return inside this iteration's rollout
            ep_rs, running = [], 0.0
            for r, d in zip(rewards, dones):
                running += r
                if d:
                    ep_rs.append(running)
                    running = 0.0
            avg_return = float(np.mean(ep_rs)) if ep_rs else float(running)
            iter_returns.append(avg_return)
            entropies.append(it_entropy / max(updates, 1))
            actor_losses.append(it_actor / max(updates, 1))
            critic_losses.append(it_critic / max(updates, 1))
            std_hist.append(float(model.log_std.exp().mean().item()))

            await run.log_metrics(
                {
                    "avg_episode_return": avg_return,
                    "policy_entropy": entropies[-1],
                    "actor_loss": actor_losses[-1],
                    "critic_loss": critic_losses[-1],
                    "exploration_std": std_hist[-1],
                },
                step=it,
            )
            if (it + 1) % 20 == 0:
                print(
                    f"  iter {it+1:3d}/{N_ITERS}  return={avg_return:8.1f}  "
                    f"std={std_hist[-1]:.3f}  entropy={entropies[-1]:.3f}"
                )
        await run.log_metric("final_avg_return", iter_returns[-1])
    return model, iter_returns, entropies, actor_losses, critic_losses, std_hist


model, iter_returns, entropies, actor_losses, critic_losses, std_hist = (
    asyncio.run(train_continuous_ppo())
)



### Checkpoint 2: the trained policy beats the measured baseline


In [ ]:
assert len(iter_returns) == N_ITERS
best_window = float(np.mean(iter_returns[-5:]))
assert best_window > random_mean + 100.0, (
    f"trained return {best_window:.1f} must clearly beat the random "
    f"baseline {random_mean:.1f}"
)
assert std_hist[-1] < std_hist[0], (
    "exploration std should shrink as the policy gains confidence "
    f"(start {std_hist[0]:.3f} -> end {std_hist[-1]:.3f})"
)
print(
    f"\n  Random baseline: {random_mean:.1f} | trained (last-5 mean): "
    f"{best_window:.1f} | exploration std: {std_hist[0]:.3f} -> {std_hist[-1]:.3f}"
)
print("\n--- Checkpoint 2 passed --- continuous PPO trained and verified\n")

register_rl_model(
    registry,
    "ppo_pendulum_continuous",
    model,
    {
        "random_baseline_return": random_mean,
        "final_window_return": best_window,
        "final_exploration_std": std_hist[-1],
    },
)



## PHASE 4 — VISUALISE: reward curve + exploration collapse


Run one episode, returning the pendulum angle at each step.


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: the dial settling down")
print("=" * 70)

# rl_diagnostic_checkpoint prints the RLDiagnostics report
rl_diagnostic_checkpoint(
    "PPO (continuous) on Pendulum-v1",
    "ppo",
    iter_returns,
    policy_losses=actor_losses,
    value_losses=critic_losses,
    entropies=entropies,
)

viz = create_visualizer()

fig = viz.training_history(
    metrics={
        "PPO-Pendulum return": iter_returns,
        "moving avg (5)": moving_average(iter_returns, 5),
        "random baseline": [random_mean] * len(iter_returns),
    },
    x_label="Iteration",
    y_label="Mean episode return",
)
fig.write_html(str(OUTPUT_DIR / "05_ppo_continuous_rewards.html"))
print(f"  Saved: {OUTPUT_DIR / '05_ppo_continuous_rewards.html'}")

fig_std, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, N_ITERS + 1), std_hist, "o-", color="#9C27B0")
ax.set_xlabel("PPO iteration")
ax.set_ylabel("exploration std = exp(log_std)")
ax.set_title("Exploration collapse: the dial settles as confidence grows")
ax.grid(True, alpha=0.3)
fig_std.tight_layout()
fig_std.savefig(str(OUTPUT_DIR / "05_ppo_continuous_std.png"), dpi=150)
plt.close(fig_std)
print(f"  Saved: {OUTPUT_DIR / '05_ppo_continuous_std.png'}")

# (c) Evaluation episodes: trained deterministic policy vs random policy,
# same protocol, same horizon. The honest behavioural claim at this CPU
# budget is DIRECTIONAL: the trained policy swings the pendulum measurably
# closer to upright than random flailing. Full stabilisation (holding
# angle ~ 0) needs a larger budget than 120 iterations — we say so.
def run_episode_angles(policy_fn, seed: int) -> list[float]:
    state, _ = env.reset(seed=seed)
    out = []
    for _ in range(200):
        action = policy_fn(state)
        state, _, terminated, truncated, _ = env.step(action)
        out.append(float(np.arctan2(state[1], state[0])))
        if terminated or truncated:
            break
    return out


model.eval()


def trained_policy(state: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        dist, _ = model(torch.from_numpy(state.astype(np.float32)).to(device))
        return dist.mean.cpu().numpy()


angles = run_episode_angles(trained_policy, seed=2026)
rng = np.random.default_rng(2026)
angles_random = run_episode_angles(
    lambda s: rng.uniform(-ACT_LIMIT, ACT_LIMIT, size=act_dim).astype(np.float32),
    seed=2026,
)
torques = []
state, _ = env.reset(seed=2026)
for _ in range(200):
    with torch.no_grad():
        dist, _ = model(torch.from_numpy(state.astype(np.float32)).to(device))
    action = dist.mean.cpu().numpy()
    state, _, terminated, truncated, _ = env.step(action)
    torques.append(float(action[0]))
    if terminated or truncated:
        break

fig_ep, (ax_a, ax_t) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax_a.plot(angles, color="#2196F3", label="trained policy (mean action)")
ax_a.plot(angles_random, color="gray", alpha=0.7, label="random policy")
ax_a.axhline(0.0, color="green", linestyle="--", alpha=0.6,
             label="upright (angle 0)")
ax_a.set_ylabel("pendulum angle (rad)")
ax_a.legend(fontsize=9)
ax_a.grid(True, alpha=0.3)
ax_t.plot(torques, color="#F44336")
ax_t.set_ylabel("applied torque (trained)")
ax_t.set_xlabel("timestep")
ax_t.grid(True, alpha=0.3)
fig_ep.suptitle("Evaluation episode: trained vs random (deterministic mean policy)")
fig_ep.tight_layout()
fig_ep.savefig(str(OUTPUT_DIR / "05_ppo_continuous_episode.png"), dpi=150)
plt.close(fig_ep)
print(f"  Saved: {OUTPUT_DIR / '05_ppo_continuous_episode.png'}")

# ── Checkpoint 3: artefacts + trained beats random on ANGLE, not just reward
import os

for artefact in (
    "05_ppo_continuous_rewards.html",
    "05_ppo_continuous_std.png",
    "05_ppo_continuous_episode.png",
):
    assert os.path.exists(OUTPUT_DIR / artefact), f"Missing: {artefact}"
late_trained = float(np.mean(np.abs(angles[-50:])))
late_random = float(np.mean(np.abs(angles_random[-50:])))
assert late_trained < late_random * 0.9, (
    f"trained policy should swing measurably closer to upright than random "
    f"(late-episode mean |angle|: trained {late_trained:.2f} rad vs random "
    f"{late_random:.2f} rad)"
)
print(
    f"\n  Late-episode mean |angle| from upright: trained "
    f"{late_trained:.2f} rad vs random {late_random:.2f} rad. At this CPU "
    "budget the policy has learned the swing-up, not full stabilisation — "
    "the direction is the verified claim; the remaining budget is stated."
)
print("\n--- Checkpoint 3 passed --- behaviour verified, not just rewards\n")



## PHASE 5 — APPLY: HVAC damper control at a data centre


DAMPER CONTROL — GOVERNANCE REPORT (measured, this run):

    Environment analogue:     Pendulum-v1 torque in [-{ACT_LIMIT}, {ACT_LIMIT}]
    Random-schedule baseline: {random_mean:.1f} mean return
    Trained policy:           {best_window:.1f} (last-5-iteration mean)
    Exploration std:          {std_hist[0]:.3f} -> {std_hist[-1]:.3f}
      (the dial's experimental wiggle, shrinking as confidence grows)

  TWO GOVERNANCE QUESTIONS, TWO MEASURED ANSWERS:
    "Better than the naive schedule?"  reward vs the measured baseline
    "Has it stopped experimenting?"    the std schedule, not a promise

  STAKEHOLDER-READY OUTPUT:
    "The continuous policy measurably outperforms a random damper
    schedule: mean return improved from {random_mean:.0f} to
    {best_window:.0f}, the exploration noise narrowed from
    {std_hist[0]:.2f} to {std_hist[-1]:.2f}, and in a head-to-head
    evaluation episode the trained controller held the process
    {late_trained:.2f} rad from setpoint versus {late_random:.2f} rad
    for the random schedule. At this training budget it has learned the
    swing-up, not full stabilisation — the remaining budget is an
    infrastructure decision, and both curves are in the tracker."


In [ ]:
# SCENARIO (anonymised, illustrative): a tropical data centre tunes its
# cooling dampers. The damper is a DIAL (0-100% open), not a switch, and
# the cost of oscillation is mechanical wear. A discrete-action agent can
# only bang the damper between preset positions; a Gaussian PPO policy
# emits a continuous setpoint and — because the exploration noise is
# measured — the facilities team can SEE when the policy stopped
# experimenting.
#
# The std schedule is the operational artefact: "experimentation ended at
# iteration ~15" is a governance statement, not a vibe. Combined with the
# reward curve it answers the two questions a facility manager asks:
# "is it better than the naive schedule?" (reward vs baseline) and "has
# it stopped fiddling?" (std collapse).

print("=" * 70)
print("  PHASE 5 — APPLY: continuous setpoint control, governed")
print("=" * 70)
print(
)



### Checkpoint 4: the governance numbers are the measured ones


In [ ]:
assert best_window > random_mean, "trained must beat random"
assert std_hist[-1] < std_hist[0], "std must shrink"
print("--- Checkpoint 4 passed --- governed application demonstrated\n")

env.close()
await conn.close()



## REFLECTION


THEORY:
  [x] Buttons vs dials: Categorical cannot express a continuous setpoint;
      the Gaussian policy emits mu(s) with a learned spread log_std
  [x] PPO's machinery is distribution-agnostic: same GAE, same clipped
      ratio, same entropy bonus — only Normal replaced Categorical

  BUILD + TRAIN:
  [x] GaussianActorCritic: tanh-squashed mean torque in [-{ACT_LIMIT}, {ACT_LIMIT}],
      free log_std parameter, separate critic ({_n_params:,} params)
  [x] {N_ITERS} iterations x {STEPS_PER_ITER} on-policy steps on Pendulum-v1,
      ExperimentTracker receipts (incl. exploration_std per iteration)
  [x] Measured: random baseline {random_mean:.1f} -> trained
      {best_window:.1f}; std {std_hist[0]:.3f} -> {std_hist[-1]:.3f}

  VISUALISE (the proof):
  [x] Reward curve with the random baseline drawn as a line
  [x] Exploration-collapse curve (std {std_hist[0]:.3f} -> {std_hist[-1]:.3f})
  [x] Eval episode, trained vs random: late-episode mean |angle|
      {late_trained:.2f} vs {late_random:.2f} rad — the swing-up is
      learned at this budget; full stabilisation is stated as remaining
      work, not claimed

  APPLY:
  [x] HVAC damper framing: "better than naive?" + "stopped
      experimenting?" answered by measured curves, not assurances
  [x] Stated limit: the std schedule is the governance artefact — when
      it has not collapsed, the policy has not finished learning

  KEY INSIGHT: Moving from discrete to continuous actions changed ONE
  object — the distribution — and left the entire PPO stack untouched.
  That is the sign of a well-factored algorithm: the policy-gradient
  theorem does not care whether actions are buttons or dials. What
  changes is what you must GOVERN: with a dial, the exploration noise
  itself is a first-class, measurable quantity.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

